In [65]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium

In [66]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random

import gymnasium as gym

from IPython.display import display, clear_output

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf

In [67]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [68]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [69]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [70]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                return result
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [71]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [72]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['ATR_14'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [73]:
train_candles = fetch_candle_data(100)
train_df = pd.DataFrame(train_candles['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

#train_df = fetch_train_candle_data(25)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

12627
12627


In [74]:
class MarketAnalysisAgent:
    def __init__(self, train_df):
        self.train_df = train_df.copy()

    def preprocess_data(self):
        ist = timezone('Asia/Kolkata')

        self.train_df['datetime'] = pd.to_datetime(self.train_df['datetime'], unit='s')
        self.train_df['datetime'] = (
            self.train_df['datetime']
            .dt.tz_localize('UTC')
            .dt.tz_convert(ist)
            .dt.tz_localize(None)
        )

        if self.train_df['datetime'].duplicated().any() or self.train_df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        self.train_df.set_index(self.train_df['datetime'], inplace=True)
        if (self.train_df['volume'] == 0).all() or self.train_df['volume'].isnull().all():
            self.train_df.drop(['volume'], axis=1, inplace=True, errors='ignore')
        self.train_df.drop(['datetime'], axis=1, inplace=True, errors='ignore')

        # Adding new columns (hour_of_day, day_of_week, etc.)
        self.train_df['hour_of_day'] = self.train_df.index.hour
        self.train_df['day_of_week'] = self.train_df.index.dayofweek

        self.train_df['high_low_range'] = self.train_df['high'] - self.train_df['low']
        self.train_df['open_close_range'] = self.train_df['open'] - self.train_df['close']

        lengths = [14]

        # Calculate technical indicators (EMA, RSI, ATR, etc.)
        for length in lengths:
            self.train_df[f'EMA_{length}'] = ta.ema(self.train_df['close'], length=length)
            self.train_df[f'RSI_{length}'] = ta.rsi(self.train_df['close'], length=length)
            self.train_df[f'ATR_{length}'] = ta.atr(
                self.train_df['high'], self.train_df['low'], self.train_df['close'], length=length
            )

        # Define target and stop loss using ATR
        self.train_df['Target'] = self.train_df['ATR_14'] * 2
        self.train_df['Stop Loss'] = self.train_df['ATR_14']

        # Round the values and remove any remaining NaN values
        self.train_df = self.train_df.round(2)
        self.train_df.dropna(inplace=True)

        return self.train_df

    def analyze(self):
        # Preprocess and analyze the data
        self.preprocess_data()
        return self.train_df

test_market_analysis_agent = MarketAnalysisAgent(train_df).analyze()
test_market_analysis_agent

,open,high,low,close,hour_of_day,day_of_week,high_low_range,open_close_range,EMA_14,RSI_14,ATR_14,Target,Stop Loss
datetime,,,,,,,,,,,,,
2024-09-06 09:43:00,51234.35,51255.65,51229.05,51239.90,9,4,26.60,-5.55,51282.65,39.21,48.97,97.94,48.97
2024-09-06 09:45:00,51237.55,51239.05,51147.20,51165.90,9,4,91.85,71.65,51267.09,28.67,53.63,107.25,53.63
2024-09-06 09:47:00,51162.00,51187.90,51143.25,51178.45,9,4,44.65,-16.45,51255.27,32.01,52.70,105.41,52.70
2024-09-06 09:49:00,51176.10,51189.25,51144.35,51151.25,9,4,44.90,24.85,51241.40,28.86,51.92,103.85,51.92
2024-09-06 09:51:00,51147.55,51164.40,51069.45,51071.00,9,4,94.95,76.55,51218.68,21.98,56.10,112.19,56.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-13 15:21:00,53592.75,53620.80,53589.90,53613.95,15,4,30.90,-21.20,53566.22,67.77,37.71,75.42,37.71
2024-12-13 15:23:00,53613.40,53638.85,53612.15,53627.90,15,4,26.70,-14.50,53574.44,69.46,36.92,73.84,36.92
2024-12-13 15:25:00,53622.55,53627.85,53599.90,53616.40,15,4,27.95,6.15,53580.04,66.37,36.28,72.57,36.28


In [75]:
class SignalGenerationAgent:
    def __init__(self, train_df):
        self.train_df = train_df.copy()

    def label_signals(self):
        # Initialize the 'Signal' column with default values
        self.train_df['Signal'] = 0 # Hold Signal
        self.train_df['Entry Price'] = 0.0
        self.train_df['Exit Price'] = 0.0

        for i in range(len(self.train_df)):
            entry_price = self.train_df['close'].iloc[i]
            target = self.train_df['Target'].iloc[i]
            stop_loss = self.train_df['Stop Loss'].iloc[i]

            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss
            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            future_data = self.train_df.iloc[i + 1:]

            for j in range(len(future_data)):
                future_high = future_data['high'].iloc[j]
                future_low = future_data['low'].iloc[j]

                if future_high >= buy_target_price:
                    self.train_df.at[self.train_df.index[i], 'Signal'] = 1 # Buy Signal
                    self.train_df.at[self.train_df.index[i], 'Entry Price'] = entry_price
                    self.train_df.at[self.train_df.index[i], 'Exit Price'] = future_high
                    break
                elif future_low <= buy_sl_price:
                    break

            for j in range(len(future_data)):
                future_high = future_data['high'].iloc[j]
                future_low = future_data['low'].iloc[j]

                if future_low <= sell_target_price:
                    self.train_df.at[self.train_df.index[i], 'Signal'] = 2 # Sell Signal
                    self.train_df.at[self.train_df.index[i], 'Entry Price'] = entry_price
                    self.train_df.at[self.train_df.index[i], 'Exit Price'] = future_low
                    break
                elif future_high >= sell_sl_price:
                    break

        return self.train_df

    def final_labelled_signals(self):
        # Remove the 'Entry Price' and 'Exit Price' columns
        self.train_df = self.label_signals()
        self.train_df = self.train_df.drop(columns=['Entry Price', 'Exit Price'])
        return self.train_df

test_signal_generation_agent = SignalGenerationAgent(test_market_analysis_agent).label_signals()
test_signal_generation_agent

,open,high,low,close,hour_of_day,day_of_week,high_low_range,open_close_range,EMA_14,RSI_14,ATR_14,Target,Stop Loss,Signal,Entry Price,Exit Price
datetime,,,,,,,,,,,,,,,,
2024-09-06 09:43:00,51234.35,51255.65,51229.05,51239.90,9,4,26.60,-5.55,51282.65,39.21,48.97,97.94,48.97,2,51239.90,51069.45
2024-09-06 09:45:00,51237.55,51239.05,51147.20,51165.90,9,4,91.85,71.65,51267.09,28.67,53.63,107.25,53.63,2,51165.90,51053.75
2024-09-06 09:47:00,51162.00,51187.90,51143.25,51178.45,9,4,44.65,-16.45,51255.27,32.01,52.70,105.41,52.70,2,51178.45,51069.45
2024-09-06 09:49:00,51176.10,51189.25,51144.35,51151.25,9,4,44.90,24.85,51241.40,28.86,51.92,103.85,51.92,2,51151.25,51030.25
2024-09-06 09:51:00,51147.55,51164.40,51069.45,51071.00,9,4,94.95,76.55,51218.68,21.98,56.10,112.19,56.10,0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-13 15:21:00,53592.75,53620.80,53589.90,53613.95,15,4,30.90,-21.20,53566.22,67.77,37.71,75.42,37.71,0,0.00,0.00
2024-12-13 15:23:00,53613.40,53638.85,53612.15,53627.90,15,4,26.70,-14.50,53574.44,69.46,36.92,73.84,36.92,0,0.00,0.00
2024-12-13 15:25:00,53622.55,53627.85,53599.90,53616.40,15,4,27.95,6.15,53580.04,66.37,36.28,72.57,36.28,0,0.00,0.00


In [76]:
class AdaptationAgent:
    def __init__(self):
        pass

    def adapt(self, reward, state):
        pass

In [77]:
train_trade_history = []

class RiskManagementAgent:
    def __init__(self):
        self.position_active = False
        self.entry_price = None
        self.position_type = None  # "long" or "short"
        self.total_reward = 0  # To track cumulative rewards

    def manage_position(self, action, current_price, target, stop_loss):
        reward = 0
        exit_position = False
        hit_target = False  # Used for accuracy calculation

        if self.position_active:
            # Calculate the target and stop loss based on the position type
            if self.position_type == "long":
                target_price = self.entry_price + target
                stop_loss_price = self.entry_price - stop_loss
            elif self.position_type == "short":
                target_price = self.entry_price - target
                stop_loss_price = self.entry_price + stop_loss

            # Check if the target or stop loss is hit
            if self.position_type == "long" and current_price >= target_price:  # Target hit for long
                reward = 1
                exit_position = True
                hit_target = True
            elif self.position_type == "long" and current_price <= stop_loss_price:  # Stop Loss hit
                reward = -1
                exit_position = True

            if self.position_type == "short" and current_price <= target_price:  # Target hit for short
                reward = 1
                exit_position = True
                hit_target = True
            elif self.position_type == "short" and current_price >= stop_loss_price:  # Stop Loss hit for short
                reward = -1
                exit_position = True

            # Exit position after target or stop loss hit
            if exit_position:
                # Update total cumulative reward
                self.total_reward += reward

                # Store the trade details in train_trade_history
                train_trade_history.append({
                    'position_type': self.position_type,
                    'entry_price': self.entry_price,
                    'exit_price': current_price,
                    'target_price': target_price,
                    'stop_loss_price': stop_loss_price,
                    'reward': reward,
                    'hit_target': hit_target,
                    'total_reward': self.total_reward
                })

                self.position_active = False
                self.entry_price = None
                self.position_type = None

        else:
            if action == 1:  # Buy (long)
                self.position_active = True
                self.entry_price = current_price
                self.position_type = "long"

            elif action == 2:  # Sell (short)
                self.position_active = True
                self.entry_price = current_price
                self.position_type = "short"

        return reward, self.position_active, hit_target


In [78]:
class MetaRLAgent:
    def __init__(self, learning_rate=0.01, discount_factor=0.99, exploration_rate=1.0):
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.exploration_rate = exploration_rate
        self.exploration_decay = 0.995
        self.min_exploration_rate = 0.01
        self.q_table = {}  # Q-table for state-action values

    def _state_to_key(self, state):
        """Converts the state into a hashable key for Q-table."""
        return tuple(np.round(state, decimals=3))

    def act(self, state):
        state_key = self._state_to_key(state)
        # Exploration vs Exploitation
        if random.random() < self.exploration_rate:
            return random.choice([0, 1, 2])  # Random action
        return np.argmax(self.q_table.get(state_key, [0, 0, 0]))  # Best action

    def learn(self, state, action, reward, next_state, done):
        state_key = self._state_to_key(state)
        next_state_key = self._state_to_key(next_state)

        # Initialize Q-values for unseen states
        if state_key not in self.q_table:
            self.q_table[state_key] = [0, 0, 0]
        if next_state_key not in self.q_table:
            self.q_table[next_state_key] = [0, 0, 0]

        # Q-Learning update
        q_value = self.q_table[state_key][action]
        max_next_q = max(self.q_table[next_state_key])
        td_target = reward + self.discount_factor * max_next_q * (not done)
        td_error = td_target - q_value
        self.q_table[state_key][action] += self.learning_rate * td_error

        # Decay exploration rate
        self.exploration_rate = max(
            self.min_exploration_rate, self.exploration_rate * self.exploration_decay
        )

    def save_model(self, filename):
        """Saves the Q-table as a CSV."""
        q_table_df = pd.DataFrame.from_dict(self.q_table, orient="index")
        q_table_df.to_csv(filename)

    def load_model(self, filename):
        """Loads the Q-table from a CSV."""
        q_table_df = pd.read_csv(filename, index_col=0)
        self.q_table = q_table_df.to_dict(orient="index")

In [79]:
class TradingEnvironment(gym.Env):
    def __init__(self, data_df, live=False, memory_length=25, sequence_length=5):
        super().__init__()
        self.df = data_df.copy()
        self.live = live
        self.memory_length = memory_length
        self.sequence_length = sequence_length
        self.current_step = 0
        self.risk_agent = RiskManagementAgent()
        self.profit_trades = 0
        self.loss_trades = 0
        self.total_trades = 0
        self.sequence = []
        self.memory = []

        # Update observation space to include sequence and memory
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.df.shape[1] * self.sequence_length + self.memory_length,),
        )
        self.action_space = gym.spaces.Discrete(3)  # 0: Hold, 1: Buy, 2: Sell

    def reset(self):
        self.current_step = 0
        self.risk_agent = RiskManagementAgent()
        self.memory = []
        self.sequence = []
        self.profit_trades = 0
        self.loss_trades = 0
        self.total_trades = 0


        self.state = self.df.iloc[self.current_step].values
        self.sequence.append(self.state)

        # Pad initial sequence with zeros if insufficient data
        padded_sequence = np.zeros((self.sequence_length, self.df.shape[1]))
        padded_sequence[-len(self.sequence):] = self.sequence

        memory_features = np.zeros(self.memory_length)
        return np.concatenate([padded_sequence.flatten(), memory_features])

    def step(self, action):
        reward = 0
        self.current_step += 1

        if self.current_step >= len(self.df):
            done = True
            next_state = np.zeros_like(self.state)
        else:
            done = False
            current_data = self.df.iloc[self.current_step]
            current_price = current_data["close"]
            target = current_data["Target"]
            stop_loss = current_data["Stop Loss"]

            reward, position_active, hit_target = self.risk_agent.manage_position(
                action, current_price, target, stop_loss
            )

            if hit_target or reward == -1:
                self.total_trades += 1
                if reward > 0:
                    self.profit_trades += 1
                else:
                    self.loss_trades += 1

            next_state = current_data.values
            self.sequence.append(next_state)

            if len(self.sequence) > self.sequence_length:
                self.sequence.pop(0)

            # Memory features
            memory_features = (
                np.mean(self.memory, axis=0) if self.memory else np.zeros(self.memory_length)
            )

            self.memory.append(self.state)
            if len(self.memory) > self.memory_length:
                self.memory.pop(0)

            # Flatten the sequence and append memory features
            sequence_data = np.array(self.sequence)
            next_state = np.concatenate([sequence_data.flatten(), memory_features])

        info = {
            "position_active": self.risk_agent.position_active,
            "profit_trades": self.profit_trades,
            "loss_trades": self.loss_trades,
            "accuracy": (self.profit_trades / self.total_trades * 100) if self.total_trades > 0 else 0,
        }
        return next_state, reward, done, info

In [80]:
def train_agent(train_df, episodes=5):
    # Step 1: Analyze and enrich the training data
    market_analysis_agent = MarketAnalysisAgent(train_df)
    enriched_train_df = market_analysis_agent.analyze()

    signal_generation_agent = SignalGenerationAgent(enriched_train_df)
    enriched_train_df = signal_generation_agent.final_labelled_signals()

    # Step 2: Initialize the trading environment and agent
    env = TradingEnvironment(enriched_train_df)
    meta_agent = MetaRLAgent()

    # Step 3: Train the agent
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0

        while True:
            action = meta_agent.act(state)
            next_state, reward, done, info = env.step(action)
            meta_agent.learn(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward

            if done:
                break

        # Log episode progress
        print(f"Episode {episode + 1} Total Reward: {total_reward}")
        print(f"Accuracy: {info['accuracy']:.2f}%")

    # Step 4: Save the trained model
    meta_agent.save_model("meta_agent_model.csv")
    return meta_agent



train_agent(train_df)

Episode 1 Total Reward: -21
Accuracy: 41.18%
Episode 2 Total Reward: -38
Accuracy: 35.38%
Episode 3 Total Reward: -38
Accuracy: 39.20%
Episode 4 Total Reward: -53
Accuracy: 37.20%
Episode 5 Total Reward: -76
Accuracy: 34.80%


In [81]:
pd.DataFrame(train_trade_history).tail(20)

,position_type,entry_price,exit_price,target_price,stop_loss_price,reward,hit_target,total_reward
862,long,53651.75,53620.55,53711.20,53622.03,-1,False,-67
863,long,53648.75,53610.35,53704.29,53620.98,-1,False,-68
864,long,53571.55,53544.60,53622.00,53546.32,-1,False,-69
865,long,53369.95,53457.50,53448.78,53330.54,1,True,-68
866,long,53532.90,53450.55,53595.42,53501.64,-1,False,-69
867,long,53388.15,53346.70,53456.81,53353.82,-1,False,-70
868,long,53524.30,53599.25,53583.46,53494.72,1,True,-69
869,long,53393.20,53355.85,53467.59,53356.00,-1,False,-70
870,long,53523.40,53485.25,53593.15,53488.53,-1,False,-71
871,long,53531.35,53486.50,53593.80,53500.13,-1,False,-72


In [82]:
def live_agent(real_df):
    market_analysis_agent = MarketAnalysisAgent(real_df)
    enriched_real_df = market_analysis_agent.analyze()
    enriched_real_df = enriched_real_df[-1:]

    env = TradingEnvironment(enriched_real_df, live=True)
    meta_agent = MetaRLAgent()
    meta_agent.load_model("meta_agent_model.csv")

    state = env.reset()
    total_reward = 0

    while True:
        action = meta_agent.act(state)
        next_state, reward, done, info = env.step(action)
        meta_agent.learn(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        print(f"Action: {['HOLD', 'BUY', 'SELL'][action]}, Reward: {reward}, Total Reward: {total_reward}")

        if done:  # Reset environment at the end of the data
            print("End of live data stream")
            break

live_agent(train_df)

Action: BUY, Reward: 0, Total Reward: 0
End of live data stream
